# HDBSCAN baseline

HDBSCAN is the hierarchical version of DBSCAN. Instead of one fixed `eps`, it looks at every density level and keeps the clusters that last the longest. 

## 1 Setup and data

HDBSCAN uses the same input as DBSCAN, the first 5 principal components so the two density methods can be compared directly

In [1]:
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import os
import joblib
import numpy as np
import pandas as pd

RANDOM_STATE = 42
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)
pd.set_option("display.max_columns", None)

In [ ]:
data = pd.read_csv("../data/H2_Cleaned_clustering.csv")
X = data.to_numpy()
density_pca = joblib.load("../dbscan/model/density_pca.joblib")   
Xd = density_pca.transform(X)
print("bookings:", len(X), " HDBSCAN input:", Xd.shape)

bookings: 48990  HDBSCAN input: (48990, 5)


Noise bookings (label −1) are left out of the scores

In [3]:
rng = np.random.default_rng(RANDOM_STATE)
S = np.sort(rng.choice(len(X), size=10_000, replace=False))   # the fixed silhouette sample


def evaluate(labels):
    clustered = labels >= 0                                    # noise (-1) is left out of the scores
    sizes = np.bincount(labels[clustered]) if clustered.any() else np.array([0])
    scores = {"clusters": labels.max() + 1,
              "noise %": (~clustered).mean() * 100,
              "smallest cluster %": sizes.min() / len(labels) * 100}
    if labels.max() < 1:                                       # the scores need at least 2 clusters
        return scores
    s = S[clustered[S]]
    scores["silhouette"] = silhouette_score(X[s], labels[s])
    scores["davies_bouldin"] = davies_bouldin_score(X[clustered], labels[clustered])
    scores["calinski_harabasz"] = calinski_harabasz_score(X[clustered], labels[clustered])
    return scores

## 2 Baseline model

defaults: `min_cluster_size=5`
`copy=True` only silences a library warning and does not change the clustering. HDBSCAN has no random start, so no seed is needed.

In [4]:
baseline = HDBSCAN(copy=True).fit(Xd)
labels = baseline.labels_
pd.DataFrame([evaluate(labels)], index=["baseline"]).round(3)

,clusters,noise %,smallest cluster %,silhouette,davies_bouldin,calinski_harabasz
baseline,2256,36.248,0.01,0.118,1.531,163.771


In [5]:
sizes = pd.Series(labels[labels >= 0]).value_counts()
pd.DataFrame({"clusters": [len(sizes)],
              "with < 50 bookings": [(sizes < 50).sum()],
              "with >= 1% of bookings": [(sizes >= 0.01 * len(X)).sum()],
              "largest cluster %": [sizes.max() / len(X) * 100]}, index=["baseline"]).round(1)

,clusters,with < 50 bookings,with >= 1% of bookings,largest cluster %
baseline,2256,2214,3,2.1


## 3 Why fragmented

Many features take only a few whole-number values (nights, guests, requests, repeat guest, segment), so the bookings sit on a grid. At each combination of those values there is a small bundle of bookings that differ only a little in LeadTime and ADR. With `min_cluster_size=5`, every bundle of 5 or more bookings can become its own cluster

The check: in how many of the tiny clusters (< 50 bookings) are all whole number features and the segment the same for every booking?

In [6]:
grid_cols = ["TotalOfSpecialRequests", "IsRepeatedGuest", "TotalGuests", "TotalNights",
             "Complementary", "Corporate", "Direct", "Groups", "Offline TA/TO", "Online TA"]
tiny = np.isin(labels, sizes[sizes < 50].index)
same = data[tiny][grid_cols].groupby(labels[tiny]).nunique().max(axis=1) == 1
print("tiny clusters:", len(same), " all whole-number features and segment the same: %.1f%%" % (same.mean() * 100))

tiny clusters: 2214  all whole-number features and segment the same: 54.3%


In 54.3% of the 2,214 tiny clusters, every booking has the same whole-number features and segment, so these clusters are just single points of the grid

## Save the baseline model

In [ ]:
joblib.dump(baseline, f"{MODEL_DIR}/hdbscan_baseline.joblib")      

['model/hdbscan_baseline.joblib']